# Dashboard consumo de sustancias e incidencias – colegios municipales DMQ

**Reproducibilidad del producto HTML.** Este cuaderno recalcula, desde la base de datos original, todas las cifras que aparecen en el HTML (texto, tablas, barras, KPI).

**Cómo usarlo:** *Entorno de ejecución → Ejecutar todo*. Cuando lo pida, sube: `base_consumo_final.dta`, `incidencias_500m.csv` (y, para la verificación final, `dash2.html`).

Resultado: `salida/app_data.json` (los agregados que incrusta el HTML) y `salida/cifras_html.csv` (una fila por cifra).

In [ ]:
# Instalación y carga de datos
import sys, os, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "numpy", "scipy"], check=False)

DATA_FILES = ["base_consumo_final.dta", "incidencias_500m.csv"]
OUT = "salida"; os.makedirs(OUT, exist_ok=True)

try:
    from google.colab import files as _colab_files   # solo existe en Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

faltan = [f for f in DATA_FILES if not os.path.exists(f)]
if faltan and IN_COLAB:
    print("Sube estos archivos:", faltan)
    subidos = _colab_files.upload()          # alternativa: montar Drive y ajustar DATA_PATH
    faltan = [f for f in DATA_FILES if not os.path.exists(f)]
if faltan:
    raise FileNotFoundError(f"Faltan archivos de datos: {faltan}")
DATA_PATH = DATA_FILES[0]
print("Datos listos:", DATA_FILES)

Dashboard de consumo de sustancias — colegios municipales (DMQ)
Este script calcula TODAS las cifras del dashboard HTML (diccionario `A` = APP_DATA).
Base: encuesta de factores de riesgo y protección asociados al consumo de sustancias,
estudiantes de EGB y Bachillerato del DMQ, período lectivo 2025-2026 (OMSCGR / INEC).
Alcance: únicamente unidades educativas con sostenimiento MUNICIPAL (variable e2).
Todas las estimaciones usan el factor de expansión `fexp`.

In [ ]:
# Configuración y utilidades
import json, re, math
import numpy as np, pandas as pd

try:
    DATA_PATH
except NameError:
    DATA_PATH = "base_consumo_final.dta"
try:
    OUT
except NameError:
    OUT = "salida"

SOSTENIMIENTO = "MUNICIPAL"
W = "fexp"

NOMBRES = {  # nombre corto y parroquia legible (la base trae nombres en mayúsculas sin tildes)
    "17H00003": "Sebastián de Benalcázar", "17H00139": "Cotocollao", "17H00244": "Humberto Mata Martínez",
    "17H00740": "Antonio José de Sucre", "17H00758": "Fernández Madrid", "17H00872": "Eugenio Espejo",
    "17H00884": "Nueve de Octubre", "17H01031": "Juan Wisneth", "17H01275": "Oswaldo Lombeyda",
    "17H01311": "Julio Enrique Moreno", "17H01395": "Quitumbe", "17H01444": "Manuel Cabeza de Vaca",
    "17H01505": "Julio Moreno Peñaherrera", "17H01658": "Calderón", "17H01806": "San Francisco de Quito",
    "17H01938": "Pedro Pablo Traversari", "17H02077": "Alfredo Pérez Guerrero", "17H02084": "Rafael Alvarado",
    "17H02762": "Experimental del Milenio Bicentenario", "17H03342": "José Ricardo Chiriboga V.",
}
PARROQ = {
    "INAQUITO": "Iñaquito", "PONCEANO": "Ponceano", "JIPIJAPA": "Jipijapa", "CENTRO HISTORICO": "Centro Histórico",
    "POMASQUI": "Pomasqui", "BELISARIO QUEVEDO": "Belisario Quevedo", "TURUBAMBA": "Turubamba",
    "GUAMANI": "Guamaní", "QUITUMBE": "Quitumbe", "ALANGASI": "Alangasí", "AMAGUANA": "Amaguaña",
    "CALDERON (CARAPUNGO)": "Calderón (Carapungo)", "GUAYLLABAMBA": "Guayllabamba", "PINTAG": "Pintag",
    "SAN JOSE DE MINAS": "San José de Minas", "TUMBACO": "Tumbaco", "CHIMBACALLE": "Chimbacalle",
}

df = None  # se asigna al cargar

def S(c):
    return df[c].astype(str).str.strip().replace({"nan": "", "None": "", "<NA>": ""})

def CODE(c):
    """Código numérico inicial de una opción tipo '3. Seguro/a' (NaN si no tiene)."""
    return pd.to_numeric(S(c).str.extract(r"^(\d+)\s*[\.\-_]")[0], errors="coerce")

def YN(c):
    """1/0 para Sí/No (NaN = en blanco, 'No sabe' o 'Prefiero no responder')."""
    s = S(c)
    yes = s.str.match(r"^(1\s*\.\s*S|S[ií]$|1(\.0)?$)")
    no = s.str.match(r"^(2\s*\.\s*N|0\s*\.\s*N|No$|0(\.0)?$)")
    return yes.astype(float).where(yes | no)

def PREV(c):
    """Prevalencia sobre TODOS los encuestados: en blanco / No / No sabe cuentan como 0."""
    return YN(c).fillna(0.0)

def OPT(c, codes, excl=(9,)):
    cd = CODE(c)
    valid = cd.notna() & ~cd.isin(excl)
    return cd.isin(codes).astype(float).where(valid)

def ANYOPT(cols, codes, excl=(5, 6, 9)):
    """1 si alguna de las columnas tiene un código en `codes`; válido si alguna tiene código válido."""
    cds = pd.concat([CODE(c) for c in cols], axis=1)
    valid_any = (cds.notna() & ~cds.isin(excl)).any(axis=1)
    hit = cds.isin(codes).any(axis=1)
    return hit.astype(float).where(valid_any)

def ANYYN(cols):
    v = pd.concat([YN(c) for c in cols], axis=1)
    return v.max(axis=1, skipna=True).where(v.notna().any(axis=1))

def DUM(c):
    return pd.to_numeric(df[c], errors="coerce")

In [ ]:
# Carga, filtro por sostenimiento y conciliación
IND = []

def add(key, label, tab, sec, dir_, cols, fn, kind="pct", fam=None, note=None):
    IND.append(dict(key=key, label=label, tab=tab, sec=sec, dir=dir_, cols=cols, fn=fn, kind=kind,
                    fam=fam or key.split("_")[0], note=note))

def TXT(c, pred):
    s = S(c).str.lower()
    return pred(s).astype(float).where(s != "")

In [ ]:
# Catálogo de indicadores
# --- Consumo de sustancias (vida / 12 meses / 30 días): base = todos los encuestados ---
SUBS = [
    ("alcohol", "Alcohol", "lic", "al1", "al4", "al5"),
    ("tabaco", "Tabaco (cigarrillo)", "lic", "ta1", "ta4", "ta5"),
    ("vapenic", "Vaper*", "lic", "ce1", "ce3", "ce4"),
    ("vapesab", "Vaper de sabores", "lic", "ce11", "ce13", "ce14"),
    ("marihuana", "Marihuana", "ilic", "ma1", "ma4", "ma8"),
    ("vapecan", "Vapeo de cannabis", "ilic", "ce6", "ce8", "ce9"),
    ("cocaina", "Cocaína", "ilic", "co1", "co4", "co6"),
    ("pastabase", "Pasta base", "ilic", "pb1", "pb4", "pb6"),
    ("extasis", "Éxtasis", "ilic", "ex1", "ex4", "ex6"),
    ("inhalables", "Inhalables", "ilic", "in1", "in4", "in7"),
    ("popper", "Popper", "ilic", "po1", "po4", "po6"),
    ("lsd", "LSD", "ilic", "ls1", "ls4", "ls6"),
    ("heroina", "Heroína", "ilic", "h1", "h4", "h6"),
    ("crack", "Crack", "ilic", "cr1", "cr4", "cr6"),
    ("hachis", "Hachís", "ilic", "ha1", "ha4", "ha6"),
    ("tranq", "Tranquilizantes sin prescripción", "med", "tr2", "tr5", "tr8"),
    ("estim", "Estimulantes sin prescripción", "med", "es2", "es5", "es8"),
    ("analg", "Analgésicos opioides sin prescripción", "med", "an2", "an5", "an8"),
]
PERIODOS = [("vida", "Alguna vez"), ("anio", "Últimos 12 meses"), ("mes", "Últimos 30 días")]
for sid, lab, grp, c1, c2, c3 in SUBS:
    for (p, plab), c in zip(PERIODOS, (c1, c2, c3)):
        add(f"{sid}_{p}", f"{lab} — {plab.lower()}", "consumo", grp, "risk", [c], (lambda c=c: PREV(c)), fam=sid)

for p, plab in PERIODOS:
    suf = {"vida": "", "anio": "_anual", "mes": "_men" if p == "mes" else "_mes"}[p]
nm_lic = {"vida": "ind_licitas", "anio": "ind_licitas_anual", "mes": "ind_licitas_men"}
nm_ilic = {"vida": "ind_ilicitas", "anio": "ind_ilicitas_anual", "mes": "ind_ilicitas_mes"}
for p, plab in PERIODOS:
    add(f"lic_{p}", f"Alguna sustancia lícita — {plab.lower()}", "consumo", "resumen", "risk", [nm_lic[p]],
        (lambda c=nm_lic[p]: DUM(c)), fam="lic")
    add(f"ilic_{p}", f"Alguna sustancia ilícita — {plab.lower()}", "consumo", "resumen", "risk", [nm_ilic[p]],
        (lambda c=nm_ilic[p]: DUM(c)), fam="ilic")

add("binge_2sem", "Consumo episódico excesivo de alcohol (5 tragos o más en una salida, últimas 2 semanas)", "consumo", "patrones", "risk",
    ["al7"], lambda: (CODE("al7").between(2, 5)).astype(float), fam="binge")
add("alcohol_colegio", "Consumió alcohol en el colegio (últimos 30 días)", "consumo", "patrones", "risk",
    ["al11_5"], lambda: DUM("al11_5").fillna(0.0), fam="alcohol")
add("vapenic_frec", "Vapeo con nicotina casi todos los días o todos los días (últimos 30 días)", "consumo", "patrones", "risk",
    ["ce5"], lambda: (CODE("ce5").isin([4, 5])).astype(float), fam="vapenic")
add("tabaco_frec", "Fuma cigarrillos casi todos los días o todos los días (últimos 30 días)", "consumo", "patrones", "risk",
    ["ta6"], lambda: (CODE("ta6").isin([3, 4])).astype(float), fam="tabaco")

OD = [("od11", "Metanfetamina"), ("od12", "Anfetamina"), ("od14", "Ketamina"), ("od15", "Opio"), ("od16", "GHB (éxtasis líquido)"),
      ("od17", "Cannabinoides sintéticos"), ("od18", "Catinonas sintéticas"), ("od19", "Aminoindanos"), ("od20", "Fenciclidina (PCP)"),
      ("od21", "Fenetilaminas"), ("od22", "Piperazinas"), ("od23", "Plantas alucinógenas"), ("od24", "Esteroides anabólicos"),
      ("od25", "Productos con cafeína"), ("od26", "Lean"), ("od27", "Fentanilo")]
for c, lab in OD:
    add(f"{c}_vida", f"{lab} — alguna vez", "consumo", "otras", "risk", [c], (lambda c=c: CODE(c).isin([2, 3, 4]).astype(float)), fam=c)
    add(f"{c}_anio", f"{lab} — últimos 12 meses", "consumo", "otras", "risk", [c], (lambda c=c: CODE(c).isin([2, 4]).astype(float)), fam=c)
    add(f"{c}_mes", f"{lab} — últimos 30 días", "consumo", "otras", "risk", [c], (lambda c=c: CODE(c).isin([4]).astype(float)), fam=c)

def _age(c, lo=6, hi=19):
    return lambda: pd.to_numeric(df[c], errors="coerce").where(lambda s: s.between(lo, hi))
add("edad_alcohol", "Edad promedio de inicio — alcohol", "consumo", "edad", "prot", ["al2"], _age("al2"), kind="mean", fam="edad")
add("edad_tabaco", "Edad promedio de inicio — tabaco", "consumo", "edad", "prot", ["ta2"], _age("ta2"), kind="mean", fam="edad")
add("edad_vapenic", "Edad promedio de inicio — vaper*", "consumo", "edad", "prot", ["ce2"], _age("ce2"), kind="mean", fam="edad")
add("edad_marihuana", "Edad promedio de inicio — marihuana", "consumo", "edad", "prot", ["ma2"], _age("ma2"), kind="mean", fam="edad")

# --- Acceso, oferta y curiosidad ---
FACIL = [("alcohol", "Alcohol", "al12"), ("marihuana", "Marihuana", "ma14"), ("cocaina", "Cocaína", "co8"),
         ("pastabase", "Pasta base", "pb8"), ("extasis", "Éxtasis", "ex8"), ("heroina", "Heroína", "h8"),
         ("crack", "Crack", "cr8"), ("tranq", "Tranquilizantes sin prescripción", "tr9"),
         ("estim", "Estimulantes sin prescripción", "es9"), ("analg", "Analgésicos sin prescripción", "an9")]
for sid, lab, c in FACIL:
    add(f"facil_{sid}", f"Le resultaría fácil conseguir: {lab}", "acceso", "facil", "risk", [c],
        (lambda c=c: OPT(c, [1])), fam="facil")
OFERTA = [("marihuana", "marihuana", "ma15"), ("cocaina", "cocaína", "co9"), ("pastabase", "pasta base", "pb9"), ("extasis", "éxtasis", "ex9")]
for sid, lab, c in OFERTA:
    add(f"oferta_{sid}_alguna", f"Le han ofrecido {lab} alguna vez", "acceso", "oferta", "risk", [c], (lambda c=c: OPT(c, [2, 3, 4])), fam="oferta")
    add(f"oferta_{sid}_30d", f"Le ofrecieron {lab} en los últimos 30 días", "acceso", "oferta", "risk", [c], (lambda c=c: OPT(c, [2])), fam="oferta")
add("oferta_mari_colegio", "Oferta de marihuana en el colegio o sus alrededores (entre quienes recibieron una oferta)", "acceso", "oferta", "risk",
    ["ma15", "ma16"], lambda: CODE("ma16").isin([2, 3]).astype(float).where(CODE("ma15").isin([2, 3, 4]) & CODE("ma16").notna()), fam="oferta")
add("curios_curioso", "Sintió curiosidad por probar alguna droga ilegal (sí o puede ser)", "acceso", "curiosidad", "risk", ["p13"], lambda: OPT("p13", [1, 3]), fam="curios")
add("curios_probaria", "Probaría drogas ilegales si tuviera la oportunidad (sí o puede ser)", "acceso", "curiosidad", "risk", ["p14"], lambda: OPT("p14", [1, 3]), fam="curios")
add("ent_drogas_colegio", "Ha visto o le han contado que hay drogas dentro del colegio", "acceso", "entorno", "risk", ["p77"], lambda: YN("p77"), fam="ent")
add("ent_drogas_alred", "Ha visto o le han contado que hay drogas en los alrededores del colegio", "acceso", "entorno", "risk", ["p78"], lambda: YN("p78"), fam="ent")
add("ent_venta_colegio", "Ha visto o le han contado que un estudiante vende o pasa drogas dentro del colegio", "acceso", "entorno", "risk", ["p79"], lambda: YN("p79"), fam="ent")
add("ent_venta_alred", "Ha visto o le han contado que un estudiante vende o pasa drogas en los alrededores", "acceso", "entorno", "risk", ["p80"], lambda: YN("p80"), fam="ent")

# --- Percepción de riesgo ---
RIESGO = ["Fumar cigarrillo frecuentemente (1 o más veces al mes)", "Fumar una o más cajetillas por día",
          "Vapear nicotina alguna vez (no más de 5 veces al año)", "Vapear nicotina frecuentemente (1 o más veces al mes)",
          "Tomar bebidas alcohólicas de vez en cuando (no más de 5 veces al año)", "Tomar cinco tragos o más, una o dos veces cada fin de semana",
          "Embriagarse con bebidas alcohólicas", "Tomar medicamentos sin prescripción frecuentemente",
          "Inhalar pegamentos, pinturas o solventes alguna vez", "Inhalar pegamentos, pinturas o solventes frecuentemente",
          "Fumar marihuana alguna vez (no más de 5 veces al año)", "Fumar marihuana frecuentemente (1 o más veces al mes)",
          "Vapear cannabis alguna vez", "Vapear cannabis frecuentemente", "Fumar pasta base alguna vez", "Fumar pasta base frecuentemente",
          "Aspirar cocaína alguna vez", "Aspirar cocaína frecuentemente", "Usar éxtasis alguna vez", "Usar éxtasis frecuentemente"]
for i, lab in enumerate(RIESGO, 1):
    add(f"riesgo_gran_r{i}", f"Gran riesgo: {lab}", "riesgo", "riesgo", "prot", [f"r{i}"], (lambda i=i: OPT(f"r{i}", [4], excl=())), fam="riesgo")
    add(f"riesgo_bajo_r{i}", f"Ningún riesgo o riesgo leve: {lab}", "riesgo", "riesgo", "risk", [f"r{i}"], (lambda i=i: OPT(f"r{i}", [1, 2], excl=())), fam="riesgob")

# --- Familia ---
add("fam_sabe", "Sus adultos casi siempre saben dónde está al salir del colegio o en fines de semana", "familia", "supervision", "prot", ["p15"], lambda: OPT("p15", [3]), fam="fam")
add("fam_superv", "Sus adultos supervisan lo que ve en TV, internet o redes", "familia", "supervision", "prot", ["p16"], lambda: YN("p16"), fam="fam")
add("fam_atentos", "Sus adultos están atentos (mucho o bastante) a lo que hace en el colegio", "familia", "supervision", "prot", ["p17"], lambda: OPT("p17", [1, 2]), fam="fam")
add("fam_comida", "Come con sus adultos 5 o más días a la semana", "familia", "supervision", "prot", ["p18"], lambda: OPT("p18", [5, 6, 7], excl=()), fam="fam")
add("fam_hora", "Sus adultos controlan la hora a la que llega en fines de semana", "familia", "supervision", "prot", ["p19"], lambda: YN("p19"), fam="fam")
add("fam_pregunta", "Sus adultos le preguntan adónde va cuando sale", "familia", "supervision", "prot", ["p20"], lambda: YN("p20"), fam="fam")
add("fam_amigos", "Sus adultos conocen bastante o más o menos a sus amigos cercanos", "familia", "supervision", "prot", ["p21"], lambda: OPT("p21", [1, 2]), fam="fam")
add("fam_desap_alcohol", "Algún adulto se molestaría mucho si llegara con tragos de más", "familia", "norma", "prot", ["p22", "p23", "p24"], lambda: ANYOPT(["p22", "p23", "p24"], [1, 2]), fam="desap")
add("fam_desap_mari", "Algún adulto se molestaría mucho si descubriera que fuma marihuana", "familia", "norma", "prot", ["p25", "p26", "p27"], lambda: ANYOPT(["p25", "p26", "p27"], [1, 2]), fam="desap")
add("fam_desap_cig", "Algún adulto se molestaría mucho si descubriera que fuma cigarrillo", "familia", "norma", "prot", ["p28", "p29", "p30"], lambda: ANYOPT(["p28", "p29", "p30"], [1, 2]), fam="desap")
add("fam_desap_vape", "Algún adulto se molestaría mucho si descubriera que vapea", "familia", "norma", "prot", ["p31", "p32", "p33"], lambda: ANYOPT(["p31", "p32", "p33"], [1, 2]), fam="desap")
add("fam_rel_padre", "Relación buena, muy buena o excelente con el padre", "familia", "relacion", "prot", ["p34"], lambda: OPT("p34", [1, 2, 3], excl=(6,)), fam="rel")
add("fam_rel_madre", "Relación buena, muy buena o excelente con la madre", "familia", "relacion", "prot", ["p35"], lambda: OPT("p35", [1, 2, 3], excl=(6,)), fam="rel")
add("fam_rel_adultos", "Relación buena, muy buena o excelente con otros adultos del hogar", "familia", "relacion", "prot", ["p36"], lambda: OPT("p36", [1, 2, 3], excl=(6,)), fam="rel")
add("fam_rel_padres", "Relación buena, muy buena o excelente entre sus padres", "familia", "relacion", "prot", ["p37"], lambda: OPT("p37", [1, 2, 3], excl=(6,)), fam="rel")
add("fam_conversa", "Ha conversado seriamente con un adulto del hogar sobre los peligros de las drogas", "familia", "relacion", "prot", ["p38"], lambda: YN("p38"), fam="fam")
add("fam_padres_droga", "Cree que alguno de sus adultos consumió drogas de joven", "familia", "riesgo", "risk", ["p39"], lambda: YN("p39"), fam="famr")
add("fam_fuma", "Algún adulto del hogar fuma regularmente", "familia", "riesgo", "risk", ["p40"], lambda: OPT("p40", [1, 2, 3]), fam="famr")
add("fam_hermano_droga", "Un hermano u otra persona del hogar consume drogas actualmente", "familia", "riesgo", "risk", ["p41"], lambda: YN("p41"), fam="famr")
add("fam_alcohol_diario", "Algún adulto del hogar toma alcohol a diario", "familia", "riesgo", "risk", ["p42", "p43", "p44"], lambda: ANYOPT(["p42", "p43", "p44"], [4, 5], excl=(6,)), fam="famr")

# --- Escuela, pares y expectativas ---
add("esc_contento", "Va contento o bastante contento al colegio", "escuela", "escuela", "prot", ["p53"], lambda: OPT("p53", [1, 2]), fam="esc")
add("esc_pertenece", "Se siente parte del colegio", "escuela", "escuela", "prot", ["p54"], lambda: YN("p54"), fam="esc")
add("esc_prof", "Buena o muy buena relación con sus profesores", "escuela", "escuela", "prot", ["p58"], lambda: OPT("p58", [1, 2]), fam="esc")
add("esc_fuga", "Se ha fugado del colegio o no ha asistido (algunas veces o más)", "escuela", "escuela", "risk", ["p55"], lambda: OPT("p55", [2, 3, 4]), fam="escr")
add("esc_faltas", "Faltó más de 10 días completos el último año lectivo", "escuela", "escuela", "risk", ["p57"], lambda: OPT("p57", [4, 5, 6]), fam="escr")
add("esc_notas_altas", "Promedio de notas entre 9,00 y 10,00", "escuela", "escuela", "prot", ["p56"], lambda: OPT("p56", [4]), fam="notas")
add("esc_notas_bajas", "Promedio de notas menor a 7,00", "escuela", "escuela", "risk", ["p56"], lambda: OPT("p56", [1, 2]), fam="notas")
add("esc_repite", "Ha repetido uno o más cursos", "escuela", "escuela", "risk", ["p74"], lambda: OPT("p74", [2, 3]), fam="escr")
add("esc_disciplina", "Ha tenido problemas de disciplina frecuentemente", "escuela", "escuela", "risk", ["p75"], lambda: OPT("p75", [3]), fam="escr")
add("par_alcohol", "La mitad o más de sus pares toma alcohol regularmente", "escuela", "pares", "risk", ["p81"], lambda: OPT("p81", [3, 4, 5], excl=(6,)), fam="par")
add("par_mari", "La mitad o más de sus pares fuma marihuana regularmente", "escuela", "pares", "risk", ["p82"], lambda: OPT("p82", [3, 4, 5], excl=(6,)), fam="par")
add("par_cig", "La mitad o más de sus pares fuma cigarrillos regularmente", "escuela", "pares", "risk", ["p83"], lambda: OPT("p83", [3, 4, 5], excl=(6,)), fam="par")
add("par_vape", "La mitad o más de sus pares vapea regularmente", "escuela", "pares", "risk", ["p84"], lambda: OPT("p84", [3, 4, 5], excl=(6,)), fam="par")
add("par_reclamo", "Sus amigos le reclamarían si supieran que fuma marihuana", "escuela", "pares", "prot", ["p76"], lambda: OPT("p76", [1]), fam="par")
add("par_indiferente", "Sus amigos no le dirían nada si supieran que fuma marihuana", "escuela", "pares", "risk", ["p76"], lambda: OPT("p76", [3]), fam="par")
add("asp_bach", "Cree probable o muy probable terminar el bachillerato", "escuela", "expectativas", "prot", ["p62"], lambda: OPT("p62", [1, 2]), fam="asp")
add("asp_univ", "Cree probable o muy probable ingresar a la educación superior", "escuela", "expectativas", "prot", ["p63"], lambda: OPT("p63", [1, 2]), fam="asp")
add("asp_trabajo", "Cree probable o muy probable trabajar tras el colegio", "escuela", "expectativas", "neu", ["p64"], lambda: OPT("p64", [1, 2]), fam="asp")
add("asp_vida", "Cree probable o muy probable lograr la vida digna que desea", "escuela", "expectativas", "prot", ["p66"], lambda: OPT("p66", [1, 2]), fam="asp")

# --- Seguridad y victimización ---
for k, lab, c in [("ruta", "en la ruta casa–escuela", "pi1"), ("escuela", "dentro de su escuela", "pi2"),
                  ("alred", "en los alrededores de su escuela", "pi3"), ("barrio", "en su barrio", "pi4")]:
    add(f"inseg_{k}", f"Se siente inseguro o muy inseguro {lab}", "seguridad", "percepcion", "risk", [c], (lambda c=c: OPT(c, [1, 2])), fam="inseg")
for k, lab, c in [("canchas_barrio", "las canchas y áreas recreativas del barrio", "ci6"), ("canchas_esc", "las canchas de la escuela", "ci7"),
                  ("banos", "los baños de la escuela", "ci8"), ("aulas", "las aulas", "ci9"), ("verdes_esc", "las áreas verdes de la escuela", "ci10")]:
    add(f"insegesp_{k}", f"Se siente inseguro o muy inseguro en {lab}", "seguridad", "espacios", "risk", [c], (lambda c=c: OPT(c, [1, 2], excl=(5,))), fam="insegesp")
for k, lab, c in [("robo_escuela", "Víctima de robo dentro de la escuela", "pi5"), ("robo_alred", "Víctima de robo en los alrededores de la escuela", "pi6"),
                  ("robo_trayecto", "Víctima de robo en el trayecto a la escuela", "pi7"), ("robo_barrio", "Víctima de robo en el barrio", "pi8"),
                  ("agr_escuela", "Víctima de agresión o amenaza dentro de la escuela", "pi9"), ("agr_alred", "Víctima de agresión o amenaza en los alrededores", "pi10"),
                  ("agr_barrio", "Víctima de agresión o amenaza en el barrio", "pi11"), ("agr_trayecto", "Víctima de agresión o amenaza en el trayecto", "pi12")]:
    add(f"victim_{k}", f"{lab} (alguna vez)", "seguridad", "victimizacion", "risk", [c], (lambda c=c: YN(c)), fam="victim")
add("victim_robo_any", "Víctima de robo en algún lugar (escuela, alrededores, trayecto o barrio)", "seguridad", "victimizacion", "risk",
    ["pi5", "pi6", "pi7", "pi8"], lambda: ANYYN(["pi5", "pi6", "pi7", "pi8"]), fam="victim")
add("victim_agr_any", "Víctima de agresión o amenaza en algún lugar", "seguridad", "victimizacion", "risk",
    ["pi9", "pi10", "pi11", "pi12"], lambda: ANYYN(["pi9", "pi10", "pi11", "pi12"]), fam="victim")
for k, lab, c in [("escuela", "las autoridades de su escuela", "ci1"), ("policia", "la Policía", "ci2"), ("militares", "los Militares", "ci3"),
                  ("bomberos", "los Bomberos de Quito", "ci4"), ("agentes", "los Agentes de Control Metropolitano", "ci5")]:
    add(f"conf_{k}", f"Confía en que le protegerían: {lab}", "seguridad", "confianza", "prot", [c], (lambda c=c: OPT(c, [3, 4])), fam="conf")
for k, lab, c in [("policia", "Policía Nacional", "p67"), ("ffaa", "Fuerzas Armadas", "p68"), ("justicia", "Sistema de justicia", "p69"),
                  ("politicos", "Los políticos", "p70"), ("iglesia", "La iglesia", "p71"), ("medios", "Medios de comunicación", "p72"),
                  ("municipio", "Municipio de Quito", "p73")]:
    add(f"confinst_{k}", f"Confía en: {lab}", "seguridad", "confianza_inst", "prot", [c], (lambda c=c: OPT(c, [1, 2], excl=())), fam="confinst")

# --- Grupos que intimidan, armas y reclutamiento ---
add("grp_mudanza", "Él/ella o su familia se mudaron por miedo, amenazas o violencia (12 meses)", "grupos", "exposicion", "risk", ["ex1_1"], lambda: YN("ex1_1"), fam="grp")
add("grp_escuela", "Vio u oyó grupos que intimidan o realizan actividades ilegales en su escuela (12 meses)", "grupos", "exposicion", "risk", ["ex2_1"], lambda: OPT("ex2_1", [1, 2, 3]), fam="grp")
add("grp_escuela_freq", "…en su escuela: todo el tiempo o a veces", "grupos", "exposicion", "risk", ["ex2_1"], lambda: OPT("ex2_1", [1, 2]), fam="grp")
add("grp_barrio", "Vio u oyó grupos que intimidan o realizan actividades ilegales en su barrio (12 meses)", "grupos", "exposicion", "risk", ["ex3_1"], lambda: OPT("ex3_1", [1, 2, 3]), fam="grp")
add("grp_barrio_freq", "…en su barrio: todo el tiempo o a veces", "grupos", "exposicion", "risk", ["ex3_1"], lambda: OPT("ex3_1", [1, 2]), fam="grp")
add("grp_extorsion", "Oyó de extorsiones («vacunas») en su barrio (12 meses)", "grupos", "exposicion", "risk", ["ex4_1"], lambda: OPT("ex4_1", [1, 2, 3]), fam="grp")
add("grp_grafitis", "Vio grafitis, símbolos o señas de grupos ilegales (12 meses)", "grupos", "exposicion", "risk", ["ex5_1"], lambda: YN("ex5_1"), fam="grp")
add("grp_conoce_nombre", "Conoce el nombre de grupos que realizan actividades ilegales", "grupos", "exposicion", "risk", ["ex6_1"], lambda: YN("ex6_1"), fam="grp")
add("grp_conoce_par", "Conoce a alguien de su edad que está en uno de esos grupos", "grupos", "exposicion", "risk", ["ex8_12"], lambda: YN("ex8_12"), fam="grp")
add("grp_defensa", "Cree que aprender defensa personal le ayudaría a vivir más tranquilo", "grupos", "exposicion", "neu", ["ex9_13"], lambda: YN("ex9_13"), fam="grpd")
add("arm_cuchillo_esc", "Vio cuchillos o navajas en su escuela (12 meses)", "grupos", "armas", "risk", ["ex10_14"], lambda: YN("ex10_14"), fam="arm")
add("arm_fuego_esc", "Vio armas de fuego en su escuela (12 meses)", "grupos", "armas", "risk", ["ex11_15"], lambda: YN("ex11_15"), fam="arm")
add("arm_cuchillo_barrio", "Vio cuchillos o navajas en su barrio (12 meses)", "grupos", "armas", "risk", ["ex12_16"], lambda: YN("ex12_16"), fam="arm")
add("arm_fuego_barrio", "Vio armas de fuego en su barrio (12 meses)", "grupos", "armas", "risk", ["ex13_17"], lambda: YN("ex13_17"), fam="arm")
add("arm_conoce_lugar", "Conoce un lugar donde se puede conseguir un arma de fuego", "grupos", "armas", "risk", ["ex14_18"], lambda: YN("ex14_18"), fam="arm")
add("arm_ofrecieron", "Le ofrecieron un arma de fuego (12 meses)", "grupos", "armas", "risk", ["ex15_19"], lambda: YN("ex15_19"), fam="arm")
add("red_videos", "Vio en redes videos o mensajes de actividades ilegales que le parecieron divertidos (12 meses)", "grupos", "redes", "risk", ["ex18"], lambda: YN("ex18"), fam="red")
add("red_inv_act", "Recibió invitaciones por redes, juegos o chats para realizar actividades ilegales (12 meses)", "grupos", "redes", "risk", ["ex19"], lambda: YN("ex19"), fam="red")
add("red_inv_grupo", "Recibió invitaciones por redes, juegos o chats para unirse a grupos ilegales (12 meses)", "grupos", "redes", "risk", ["ex20"], lambda: YN("ex20"), fam="red")
add("rec_invitado", "Lo invitaron personalmente a ayudar o unirse a un grupo ilegal (12 meses)", "grupos", "reclutamiento", "risk", ["pg8"], lambda: YN("pg8"), fam="rec")
add("rec_presion", "Se sentiría presionado/a a unirse si un amigo muy querido se uniera", "grupos", "reclutamiento", "risk", ["pg9"], lambda: YN("pg9"), fam="rec")
add("rec_dice_no", "Logra decir NO siempre cuando le piden algo que no quiere", "grupos", "reclutamiento", "prot", ["pg1"], lambda: OPT("pg1", [1]), fam="rec")
for i, lab in enumerate(["Profesores", "Psicólogos", "Familia", "Un amigo/a", "Policía", "A nadie", "No sabe"], 1):
    add(f"acude_{i}", f"Acudiría a: {lab}", "grupos", "acude", "risk" if i == 6 else "neu", [f"pd2_{i}"], (lambda i=i: DUM(f"pd2_{i}")), fam="acude")
for i, lab in [(1, "Nada"), (2, "Dinero"), (3, "Protección / seguridad"), (4, "Amigos/as"), (5, "Respeto"), (6, "Parejas"), (7, "La familia que nunca tuve")]:
    add(f"gana_{i}", f"Cree que se gana: {lab}", "grupos", "gana", "risk" if i > 1 else "prot", [f"pd3_{i}"], (lambda i=i: DUM(f"pd3_{i}")), fam="gana")
for i, lab in [(1, "Nada"), (2, "Los estudios"), (3, "Protección / seguridad"), (4, "La familia"), (5, "El respeto"), (6, "Parejas"), (7, "Amigos/as"), (8, "La tranquilidad")]:
    add(f"pierde_{i}", f"Cree que se pierde: {lab}", "grupos", "pierde", "risk" if i == 1 else "prot", [f"pd4_{i}"], (lambda i=i: DUM(f"pd4_{i}")), fam="pierde")
for i, lab, d_ in [(1, "Poderoso/a", "risk"), (2, "Con miedo y asustado/a", "neu"), (3, "Sin ganas de ir al colegio", "neu"), (4, "Sin ganas de salir en el barrio", "neu"),
                   (5, "Presionado/a, entre la espada y la pared", "neu"), (6, "Feliz", "risk"), (7, "Importante", "risk"), (8, "Valioso/a", "risk"),
                   (9, "Tomado/a en cuenta", "risk"), (10, "Preocupado/a", "neu"), (11, "Triste", "neu"), (12, "Indiferente", "neu")]:
    add(f"siente_{i}", f"Si lo invitaran se sentiría: {lab}", "grupos", "siente", d_, [f"pd7_{i}"], (lambda i=i: DUM(f"pd7_{i}")), fam="siente")

# --- Perfil ---
add("per_mujer", "Mujeres", "perfil", "composicion", "neu", ["d1"], lambda: TXT("d1", lambda s: s.eq("mujer")), fam="per")
add("per_edad", "Edad promedio (años)", "perfil", "composicion", "neu", ["d3"], lambda: pd.to_numeric(df["d3"], errors="coerce").where(lambda s: s.between(10, 22)), kind="mean", fam="per")
add("per_9no", "9.º de EGB", "perfil", "composicion", "neu", ["e3"], lambda: TXT("e3", lambda s: s.eq("9noegb")), fam="per")
add("per_1ro", "1.º de Bachillerato", "perfil", "composicion", "neu", ["e3"], lambda: TXT("e3", lambda s: s.eq("1robu")), fam="per")
add("per_3ro", "3.º de Bachillerato", "perfil", "composicion", "neu", ["e3"], lambda: TXT("e3", lambda s: s.eq("3robu")), fam="per")
add("per_extranj", "Nacionalidad distinta a la ecuatoriana", "perfil", "composicion", "neu", ["d4"], lambda: TXT("d4", lambda s: s.ne("ecuatoriano")), fam="per")
add("per_indigena", "Se autoidentifica indígena", "perfil", "composicion", "neu", ["d6"], lambda: TXT("d6", lambda s: s.eq("indígenas")), fam="per")
add("per_afro", "Se autoidentifica afrodescendiente", "perfil", "composicion", "neu", ["d6"], lambda: TXT("d6", lambda s: s.eq("afrodescendientes")), fam="per")
add("per_idioma", "Se habla otro idioma además del español en casa", "perfil", "composicion", "neu", ["d7"], lambda: (CODE("d7") > 1).astype(float).where(CODE("d7").notna()), fam="per")
add("per_biparental", "Hogar biparental (padre y madre)", "perfil", "hogar", "neu", ["tipohogar"], lambda: TXT("tipohogar", lambda s: s.str.startswith("biparental")), fam="hog")
add("per_monoparental", "Hogar monoparental", "perfil", "hogar", "neu", ["tipohogar"], lambda: TXT("tipohogar", lambda s: s.str.startswith("monoparental")), fam="hog")
add("per_sin_adulto", "Sin núcleo parental (abuelos, otro adulto o solo hermanos/as)", "perfil", "hogar", "neu", ["tipohogar"], lambda: TXT("tipohogar", lambda s: s.str.startswith("sin ")), fam="hog")
add("per_sin_agua", "Sin acceso a agua potable por tubería en casa", "perfil", "hogar", "risk", ["d12"], lambda: 1 - YN("d12"), fam="hog")
add("per_sin_internet", "Sin servicio de internet en casa", "perfil", "hogar", "risk", ["d15"], lambda: 1 - YN("d15"), fam="hog")
add("per_internet_colegio", "Se conecta a internet solo en el colegio", "perfil", "hogar", "risk", ["d16"], lambda: TXT("d16", lambda s: s.str.startswith("solo en el colegio")), fam="hog")
add("per_falta_dinero", "En el hogar faltó dinero para necesidades básicas (12 meses)", "perfil", "hogar", "risk", ["p52"], lambda: YN("p52"), fam="hog")
add("per_migro", "Tiene familiares que migraron en los últimos 10 años por falta de trabajo", "perfil", "hogar", "neu", ["p45"], lambda: YN("p45"), fam="hog")
add("per_cuida", "Cuida a algún familiar en su día a día", "perfil", "hogar", "neu", ["p47"], lambda: YN("p47"), fam="hog")
add("per_pie", "Llega al colegio a pie", "perfil", "trayecto", "neu", ["d18"], lambda: OPT("d18", [5], excl=()), fam="tray")
add("per_publico", "Llega al colegio en transporte público", "perfil", "trayecto", "neu", ["d18"], lambda: OPT("d18", [4], excl=()), fam="tray")
add("per_auto", "Lo traen en auto o moto de alguien del hogar", "perfil", "trayecto", "neu", ["d18"], lambda: OPT("d18", [1, 2], excl=()), fam="tray")
add("per_traslado_largo", "Tarda más de una hora (ida y vuelta) en trasladarse", "perfil", "trayecto", "neu", ["d21"], lambda: TXT("d21", lambda s: s.str.startswith("más de")), fam="tray")
add("per_extraesc", "Realiza actividades extraescolares en la semana", "perfil", "trayectoria", "prot", ["d28"], lambda: YN("d28"), fam="trayect")
add("per_econ_riesgo", "Siente que su educación está en riesgo por la situación económica del hogar", "perfil", "trayectoria", "risk", ["d31"], lambda: YN("d31"), fam="trayect")
add("per_dejar", "Consideró dejar la escuela en los últimos 12 meses", "perfil", "trayectoria", "risk", ["d32"], lambda: YN("d32"), fam="trayect")
add("per_trabaja", "Trabaja regularmente además de estudiar", "perfil", "trayectoria", "risk", ["p59"], lambda: YN("p59"), fam="trayect")

print(len(IND), "indicadores definidos")

In [ ]:
# Carga de la base y filtro por sostenimiento
base_cols = ["gid", "ue", "amie", "e1", "e2", "fexp", "x", "y"]
need = sorted(set(base_cols) | {c for i in IND for c in i["cols"]})
raw_cols = pd.read_stata(DATA_PATH, iterator=True).variable_labels().keys()
faltan = [c for c in need if c not in raw_cols]
assert not faltan, f"Columnas ausentes en la base: {faltan}"
_all = pd.read_stata(DATA_PATH, columns=need, convert_categoricals=False)
N_BASE = len(_all)
df = _all[_all["e2"].astype(str).str.strip().str.upper() == SOSTENIMIENTO].reset_index(drop=True)
del _all
assert df["gid"].is_unique
w = df[W].astype(float)
CONC = [
    {"paso": "Registros en la base completa (todos los sostenimientos)", "n": int(N_BASE)},
    {"paso": "Registros de unidades educativas con sostenimiento municipal", "n": int(len(df))},
    {"paso": "Unidades educativas municipales (códigos AMIE distintos)", "n": int(df["amie"].nunique())},
    {"paso": "Población estimada con el factor de expansión (municipales)", "n": int(round(w.sum()))},
]
print(CONC)

In [ ]:
# Cálculo de indicadores por colegio y para el promedio municipal
def wmedian(v, wt):
    o = np.argsort(v.values)
    v2, w2 = v.values[o], wt.values[o]
    cw = np.cumsum(w2)
    return float(v2[np.searchsorted(cw, cw[-1] / 2.0)])

def wilson(p, n, z=1.96):
    if n == 0 or p is None or np.isnan(p):
        return None
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return [max(0.0, c - h), min(1.0, c + h)]

SER = {}
for i in IND:
    s = i["fn"]()
    SER[i["key"]] = pd.to_numeric(s, errors="coerce").astype(float).reset_index(drop=True)

AMIES = sorted(df["amie"].unique(), key=lambda a: NOMBRES[a])

# --- Incidencias en 500 m alrededor de cada colegio (capa buf_500_uni del dashboard ArcGIS) ---
try:
    INC_PATH
except NameError:
    INC_PATH = DATA_FILES[1] if "DATA_FILES" in globals() and len(DATA_FILES) > 1 else "incidencias_500m.csv"
inc = pd.read_csv(INC_PATH, encoding="utf-8")
inc = inc[inc["amie"].isin(AMIES)].reset_index(drop=True)
assert set(inc["amie"]) == set(AMIES), "Faltan colegios en la tabla de incidencias"
for _, r_ in inc.iterrows():  # control de transcripción: TOTAL = suma de ambos períodos
    assert sum(int(r_[k]) for k in ["R12", "R12_", "E12", "E12_", "L12", "L12_", "VC12", "VC12_", "H12", "H12_"]) == int(r_["TOTAL"]), r_["amie"]
INC = inc.set_index("amie")
def zname(a):
    return str(INC.loc[a, "Admin_Zona"]).strip().title().replace("De ", "de ")
ZONAS = sorted({zname(a) for a in AMIES})
ZID = {z: "Z:" + z for z in ZONAS}
UNITS = ["MUN"] + [ZID[z] for z in ZONAS] + AMIES
masks = {"MUN": pd.Series(True, index=df.index)}
for z in ZONAS:
    masks[ZID[z]] = df["amie"].isin([a for a in AMIES if zname(a) == z])
for a in AMIES:
    masks[a] = df["amie"] == a

VAL, NV, CI, ZS = {u: {} for u in UNITS}, {u: {} for u in UNITS}, {u: {} for u in UNITS}, {u: {} for u in UNITS}

def estimate(x, m, kind):
    xs, ws = x[m], w[m]
    v = xs.notna()
    n = int(v.sum())
    if n == 0:
        return None, 0
    if kind == "med":
        return wmedian(xs[v], ws[v]), n
    return float((xs[v] * ws[v]).sum() / ws[v].sum()), n

for i in IND:
    k, kind = i["key"], i["kind"]
    x = SER[k]
    for u in UNITS:
        est, n = estimate(x, masks[u], kind)
        VAL[u][k], NV[u][k] = est, n
        if kind == "pct" and est is not None:
            CI[u][k] = wilson(est, n)
    if kind == "pct":
        for a in UNITS[1:]:
            ps, ns = VAL[a][k], NV[a][k]
            pr, nr = estimate(x, ~masks[a], kind)
            if ps is None or pr is None or ns < 10 or nr < 10:
                ZS[a][k] = None
                continue
            pp = (ps * ns + pr * nr) / (ns + nr)
            se = math.sqrt(max(pp * (1 - pp), 1e-12) * (1 / ns + 1 / nr))
            ZS[a][k] = (ps - pr) / se

In [ ]:
# Incidencias y establecimientos en 500 m (suma por colegio / zona / total municipal)
INC_T = [("R", "Robos a personas", "R12"), ("E", "Escándalos", "E12"), ("L", "Libadores", "L12"),
         ("VC", "Venta de droga", "VC12"), ("H", "Homicidios", "H12")]
EST = [("BAR", "Bares"), ("DISCO", "Discotecas"), ("NIGHT", "Night clubs"), ("TIENDA", "Tiendas"), ("LICOR", "Licorerías")]
def inc_row(a):
    r_ = INC.loc[a]
    d = {}
    for t, lab, c in INC_T:
        d[f"inc_{t}_a"] = float(r_[c]); d[f"inc_{t}_p"] = float(r_[c + "_"])
    d["inc_TOT_a"] = sum(d[f"inc_{t}_a"] for t, _, _ in INC_T)
    d["inc_TOT_p"] = sum(d[f"inc_{t}_p"] for t, _, _ in INC_T)
    for c, lab in EST:
        d[f"est_{c}"] = float(r_[c])
    d["est_BARES"] = d["est_BAR"] + d["est_DISCO"] + d["est_NIGHT"]
    return d
INCV = {a: inc_row(a) for a in AMIES}
for u in UNITS:
    mem = AMIES if u == "MUN" else ([a for a in AMIES if ZID.get(zname(a)) == u] if u.startswith("Z:") else [u])
    tot = {k: sum(INCV[a][k] for a in mem) for k in INCV[AMIES[0]]}
    tot["inc_TOT_chg"] = (tot["inc_TOT_a"] / tot["inc_TOT_p"] - 1) if tot["inc_TOT_p"] else None
    for k, v in tot.items():
        VAL[u][k] = v; NV[u][k] = len(mem)
INC_CAT = []
PER = {"a": "12 meses previos al levantamiento", "p": "13 a 24 meses previos al levantamiento"}
for t, lab, _ in INC_T + [("TOT", "Total de incidencias", None)]:
    for per in ("a", "p"):
        INC_CAT.append(dict(key=f"inc_{t}_{per}", label=f"{lab} en 500 m ({PER[per]})", tab="incid", sec="incid", dir="risk", kind="cnt", fam="inc"))
INC_CAT.append(dict(key="inc_TOT_chg", label="Variación del total de incidencias en 500 m (12 meses previos vs 13 a 24 meses previos)", tab="incid", sec="incid", dir="risk", kind="chg", fam="inc"))
for c, lab in EST + [("BARES", "Bares, discotecas y night clubs")]:
    INC_CAT.append(dict(key=f"est_{c}", label=f"{lab} en 500 m", tab="incid", sec="estab", dir="risk", kind="cnt", fam="est"))

# --- Cruce encuesta × incidencias (correlación de Spearman entre los 20 colegios) ---
from scipy.stats import spearmanr
CROSS_X = ["inc_TOT_a", "inc_R_a", "inc_E_a", "inc_L_a", "inc_VC_a", "est_LICOR", "est_TIENDA", "inc_TOT_chg"]
CROSS = {}
for xk in CROSS_X:
    xv = np.array([VAL[a][xk] for a in AMIES], dtype=float)
    CROSS[xk] = {}
    for i in IND:
        if i["kind"] not in ("pct", "mean") or i["tab"] == "perfil":
            continue
        yv = np.array([VAL[a][i["key"]] if VAL[a][i["key"]] is not None else np.nan for a in AMIES], dtype=float)
        if np.isnan(yv).any() or np.nanstd(yv) == 0 or np.std(xv) == 0:
            continue
        rho, pv = spearmanr(xv, yv)
        CROSS[xk][i["key"]] = [round(float(rho), 4), round(float(pv), 4)]

In [ ]:
# Sanidad de los indicadores (cobertura de datos válidos)
cov = pd.DataFrame({"key": [i["key"] for i in IND], "n_valid": [NV["MUN"][i["key"]] for i in IND],
                    "mun": [VAL["MUN"][i["key"]] for i in IND]})
print(cov[cov.n_valid < 0.3 * len(df)].to_string())
print("Indicadores sin ningún dato válido:", cov[cov.n_valid == 0].key.tolist())

In [ ]:
# Estructura de pestañas, lecturas automáticas y APP_DATA
def pct_s(v, d=1):
    return f"{v*100:.{d}f}".replace(".", ",") + "%"

def num_s(v, d=1):
    return f"{v:.{d}f}".replace(".", ",")

def miles(n):
    return f"{int(round(n)):,}".replace(",", ".")

CAT = {i["key"]: i for i in IND}

SUBS_LIC = [("alcohol", "Alcohol"), ("tabaco", "Tabaco (cigarrillo)"), ("vapenic", "Vaper*"), ("vapesab", "Vaper de sabores")]
SUBS_ILIC = [(s[0], s[1]) for s in SUBS if s[2] == "ilic"]
SUBS_MED = [(s[0], s[1]) for s in SUBS if s[2] == "med"]
OTRAS = [(c, lab) for c, lab in OD]

def bars(title, keys, note=None):
    return {"title": title, "type": "bars", "keys": keys, "note": note}

def prev3(title, rows, note=None):
    return {"title": title, "type": "prev3", "rows": [[lab, sid] for sid, lab in rows], "note": note}

def keys_of(tab, sec):
    return [i["key"] for i in IND if i["tab"] == tab and i["sec"] == sec]

TABS = [
    {"id": "perfil", "label": "Perfil", "secs": [
        bars("Composición del alumnado encuestado", keys_of("perfil", "composicion")),
        bars("Hogar y condiciones de vida", keys_of("perfil", "hogar")),
        bars("Trayecto al colegio", keys_of("perfil", "trayecto")),
        bars("Trayectoria escolar y trabajo", keys_of("perfil", "trayectoria")),
    ]},
    {"id": "consumo", "label": "Consumo de sustancias", "secs": [
        prev3("Resumen", [("lic", "Alguna sustancia lícita"), ("ilic", "Alguna sustancia ilícita")],
              "Lícitas: alcohol, tabaco, vaper con nicotina o de sabores (indicador ind_licitas de la base). Ilícitas: indicador ind_ilicitas precalculado en la base, usado tal como viene."),
        prev3("Sustancias lícitas", SUBS_LIC, "* Vaper con nicotina."),
        prev3("Sustancias ilícitas", SUBS_ILIC),
        prev3("Medicamentos sin prescripción médica", SUBS_MED),
        bars("Patrones de consumo", ["binge_2sem", "alcohol_colegio", "vapenic_frec", "tabaco_frec"]),
        bars("Edad promedio de inicio (entre quienes consumieron)", keys_of("consumo", "edad")),
        prev3("Otras sustancias (módulo de otras drogas)", OTRAS,
              "Los valores de «Productos con cafeína» y «Lean» son mucho más altos que los del resto del módulo; se muestran tal como constan en la base."),
    ]},
    {"id": "acceso", "label": "Acceso, oferta y curiosidad", "secs": [
        bars("Facilidad percibida para conseguir la sustancia", keys_of("acceso", "facil"), "Porcentaje que respondió «me resultaría fácil»."),
        bars("Ofertas recibidas", keys_of("acceso", "oferta")),
        bars("Curiosidad y disposición a probar", keys_of("acceso", "curiosidad")),
        bars("Presencia de drogas en el entorno escolar (visto o contado)", keys_of("acceso", "entorno")),
    ]},
    {"id": "riesgo", "label": "Percepción de riesgo", "secs": [
        {"title": t, "type": "pair", "rows": [[RIESGO[n - 1], f"riesgo_gran_r{n}", f"riesgo_bajo_r{n}"] for n in rng],
         "note": "Barra: % que considera que existe «gran riesgo». Columna derecha: % que considera «ningún riesgo» o «riesgo leve»."}
        for t, rng in [("Tabaco y vapeo de nicotina", range(1, 5)), ("Alcohol", range(5, 8)),
                       ("Medicamentos sin prescripción e inhalables", range(8, 11)), ("Marihuana y vapeo de cannabis", range(11, 15)),
                       ("Pasta base, cocaína y éxtasis", range(15, 21))]
    ]},
    {"id": "familia", "label": "Familia", "secs": [
        bars("Supervisión y acompañamiento", keys_of("familia", "supervision")),
        bars("Normas: reacción esperada de los adultos", keys_of("familia", "norma")),
        bars("Calidad de las relaciones y comunicación", keys_of("familia", "relacion")),
        bars("Consumo en el entorno familiar", keys_of("familia", "riesgo")),
    ]},
    {"id": "escuela", "label": "Escuela y pares", "secs": [
        bars("Vínculo y trayectoria escolar", keys_of("escuela", "escuela")),
        bars("Grupo de pares", keys_of("escuela", "pares")),
        bars("Expectativas a futuro", keys_of("escuela", "expectativas")),
    ]},
    {"id": "seguridad", "label": "Seguridad y victimización", "secs": [
        bars("Percepción de inseguridad", keys_of("seguridad", "percepcion")),
        bars("Inseguridad en espacios concretos", keys_of("seguridad", "espacios")),
        bars("Victimización (alguna vez)", keys_of("seguridad", "victimizacion")),
        bars("Confianza en que lo protegerían si cuenta un problema", keys_of("seguridad", "confianza")),
        bars("Confianza en instituciones", keys_of("seguridad", "confianza_inst")),
    ]},
    {"id": "grupos", "label": "Grupos ilegales y reclutamiento", "secs": [
        bars("Exposición a grupos que intimidan o realizan actividades ilegales", keys_of("grupos", "exposicion")),
        bars("Armas", keys_of("grupos", "armas")),
        bars("Redes sociales", keys_of("grupos", "redes")),
        bars("Reclutamiento y capacidad de rechazo", keys_of("grupos", "reclutamiento")),
        bars("¿A quién acudiría si se siente en peligro ante un grupo ilegal?", keys_of("grupos", "acude")),
        bars("¿Qué cree que se gana al unirse a un grupo ilegal?", keys_of("grupos", "gana")),
        bars("¿Qué cree que se pierde al unirse?", keys_of("grupos", "pierde")),
        bars("¿Cómo se sentiría si lo invitaran a unirse?", keys_of("grupos", "siente")),
    ]},
]
# prev3 usa las claves {id}_vida / _anio / _mes
TABS.append({"id": "incid", "label": "Incidencias (500 m)", "hidden": True, "secs": [
    bars("Incidencias y establecimientos en 500 m", [c["key"] for c in INC_CAT])]})
HEAD = ["inc_TOT_a", "ilic_vida", "lic_vida", "alcohol_anio", "alcohol_mes", "vapenic_anio", "tabaco_anio", "marihuana_anio", "binge_2sem",
        "facil_marihuana", "oferta_marihuana_alguna", "ent_drogas_colegio", "inseg_alred", "victim_robo_any", "victim_agr_any",
        "grp_escuela", "fam_sabe", "esc_contento"]

# --- Colegios ---
SCH = []
for a in AMIES:
    sub = df[df["amie"] == a]
    SCH.append({"id": a, "name": NOMBRES[a], "full": re.sub(r"\s+", " ", str(sub["ue"].iloc[0]).strip()),
                "parroquia": PARROQ.get(str(sub["e1"].iloc[0]).strip(), str(sub["e1"].iloc[0]).strip().title()),
                "n": int(len(sub)), "N": float(w[masks[a]].sum()), "zona": ZID[zname(a)], "barrio": str(INC.loc[a, "Nombre_Bar"]).strip().title(),
                "lat": float(sub["y"].mean()), "lon": float(sub["x"].mean()),
                "small": bool(len(sub) < 100)})

# --- Lecturas automáticas por colegio (comparación con el resto de colegios municipales) ---
EXCL_FAM = {"edad", "per", "hog", "tray", "trayect", "acude", "gana", "pierde", "siente", "grpd"}
def lecturas(a):
    mun_rest = {}
    cand = []
    for i in IND:
        k = i["key"]
        if i["kind"] != "pct" or i["dir"] == "neu" or i["tab"] in ("perfil",) or i["fam"] in EXCL_FAM:
            continue
        z = ZS[a].get(k)
        ps = VAL[a][k]
        if z is None or ps is None:
            continue
        # promedio del resto de colegios municipales
        x = SER[k]
        pr, nr = estimate(x, ~masks[a], "pct")
        diff = ps - pr
        if abs(z) >= 2.58 and abs(diff) >= 0.05 and max(ps, pr) >= 0.05:
            worse = (i["dir"] == "risk" and diff > 0) or (i["dir"] == "prot" and diff < 0)
            cand.append((abs(z), k, i, ps, pr, diff, worse))
    cand.sort(key=lambda t: -t[0])
    out = {"alertas": [], "fortalezas": []}
    seen = {"alertas": set(), "fortalezas": set()}
    for _, k, i, ps, pr, diff, worse in cand:
        tipo = "alertas" if worse else "fortalezas"
        if i["fam"] in seen[tipo] or len(out[tipo]) >= 4:
            continue
        seen[tipo].add(i["fam"])
        sgn = "+" if diff > 0 else "−"
        out[tipo].append({"key": k, "txt": f"{i['label']}: {pct_s(ps)} frente a {pct_s(pr)} en el resto de colegios municipales ({sgn}{num_s(abs(diff)*100)} pp)."})
    return out

LEC = {a: lecturas(a) for a in UNITS[1:]}

def resumen(u):
    v = VAL[u]
    top = sorted([(v[f"{sid}_anio"], lab) for sid, lab, *_ in SUBS], reverse=True)[:3]
    b = [f"Sustancias con mayor consumo en los últimos 12 meses: " + ", ".join(f"{lab.replace('Vaper*','Vaper con nicotina')} ({pct_s(p)})" for p, lab in top) + ".",
         f"Alguna sustancia lícita alguna vez: {pct_s(v['lic_vida'])}; alguna sustancia ilícita alguna vez: {pct_s(v['ilic_vida'])}.",
         f"Se siente inseguro en los alrededores del colegio: {pct_s(v['inseg_alred'])}; fue víctima de robo en algún lugar: {pct_s(v['victim_robo_any'])}.",
         f"Le han ofrecido marihuana alguna vez: {pct_s(v['oferta_marihuana_alguna'])}; ha visto o le han contado que hay drogas dentro del colegio: {pct_s(v['ent_drogas_colegio'])}.",
         f"Incidencias en 500 m: {miles(v['inc_TOT_a'])} en los 12 meses previos al levantamiento" + (f" ({'+' if v['inc_TOT_chg'] > 0 else '−'}{num_s(abs(v['inc_TOT_chg'])*100)}% frente a los 12 meses anteriores)." if v['inc_TOT_chg'] is not None else ".")]
    return b
RES = {u: resumen(u) for u in UNITS}

# --- Lecturas generales (promedio municipal y entre colegios) ---
big = [s for s in SCH if not s["small"]]
def extremos(k):
    xs = sorted(big, key=lambda s: VAL[s["id"]][k])
    return xs[0], xs[-1]
lo_i, hi_i = extremos("ilic_vida")
lo_a, hi_a = extremos("alcohol_anio")
lo_s, hi_s = extremos("inseg_alred")
V = VAL["MUN"]
IDEAS = [
    {"t": "Consumo en el conjunto de colegios municipales", "b": [
        f"{pct_s(V['lic_vida'])} de los estudiantes ha probado alguna sustancia lícita y {pct_s(V['ilic_vida'])} alguna sustancia ilícita (alguna vez).",
        f"En los últimos 12 meses: alcohol {pct_s(V['alcohol_anio'])}, vaper de sabores {pct_s(V['vapesab_anio'])}, vaper con nicotina {pct_s(V['vapenic_anio'])}, tabaco {pct_s(V['tabaco_anio'])} y marihuana {pct_s(V['marihuana_anio'])}.",
        f"Consumo episódico excesivo de alcohol en las últimas 2 semanas: {pct_s(V['binge_2sem'])}."]},
    {"t": "Diferencias entre colegios", "b": [
        f"Sustancias ilícitas (alguna vez): de {pct_s(VAL[lo_i['id']]['ilic_vida'])} ({lo_i['name']}) a {pct_s(VAL[hi_i['id']]['ilic_vida'])} ({hi_i['name']}).",
        f"Alcohol (12 meses): de {pct_s(VAL[lo_a['id']]['alcohol_anio'])} ({lo_a['name']}) a {pct_s(VAL[hi_a['id']]['alcohol_anio'])} ({hi_a['name']}).",
        "Los rangos consideran colegios con 100 o más estudiantes encuestados; los de muestra menor se marcan con cautela en la ficha."]},
    {"t": "Entorno escolar y seguridad", "b": [
        f"{pct_s(V['inseg_alred'])} se siente inseguro en los alrededores del colegio (rango entre colegios: {pct_s(VAL[lo_s['id']]['inseg_alred'])}–{pct_s(VAL[hi_s['id']]['inseg_alred'])}).",
        f"{pct_s(V['ent_drogas_colegio'])} ha visto o le han contado que hay drogas dentro del colegio; {pct_s(V['oferta_marihuana_alguna'])} recibió alguna vez una oferta de marihuana.",
        f"{pct_s(V['victim_robo_any'])} fue víctima de robo y {pct_s(V['victim_agr_any'])} de agresión o amenaza en algún lugar (alguna vez)."]},
]

KPI = [
    {"num": f"{len(SCH)} · {miles(len(df))}", "lab": "colegios municipales · estudiantes encuestados"},
    {"num": pct_s(V["lic_vida"]), "lab": "ha probado alguna sustancia lícita (alguna vez)"},
    {"num": pct_s(V["ilic_vida"]), "lab": "ha probado alguna sustancia ilícita (alguna vez)"},
    {"num": pct_s(V["inseg_alred"]), "lab": "se siente inseguro en los alrededores del colegio"},
]

def r8(x):
    return None if x is None or (isinstance(x, float) and np.isnan(x)) else round(float(x), 8)

A = {
    "meta": {"base": "Encuesta de factores de riesgo y protección asociados al consumo de sustancias — DMQ, período lectivo 2025–2026",
             "alcance": "Unidades educativas con sostenimiento municipal", "n_estudiantes": int(len(df)), "N_expandido": float(round(w.sum(), 2)),
             "n_colegios": len(SCH), "min_muestra": 100},
    "conciliacion": CONC,
    "catalog": {i["key"]: {"label": i["label"], "tab": i["tab"], "sec": i["sec"], "dir": i["dir"], "kind": i["kind"], "fam": i["fam"]} for i in IND + INC_CAT},
    "zones": [{"id": ZID[z], "name": z, "schools": [a for a in AMIES if zname(a) == z], "n": int(masks[ZID[z]].sum()), "N": float(w[masks[ZID[z]]].sum())} for z in ZONAS],
    "inc_types": [{"t": t, "label": lab} for t, lab, _ in INC_T], "establecimientos": [{"c": c, "label": lab} for c, lab in EST],
    "cross": CROSS, "cross_x": CROSS_X,
    "tabs": TABS, "head": HEAD, "schools": SCH,
    "vals": {u: {k: r8(v) for k, v in VAL[u].items()} for u in UNITS},
    "nvalid": NV,
    "ci": {u: {k: ([r8(c[0]), r8(c[1])] if c else None) for k, c in CI[u].items()} for u in UNITS},
    "z": {a: {k: r8(z) for k, z in ZS[a].items()} for a in UNITS[1:]},
    "lecturas": LEC, "resumen": RES, "ideas": IDEAS, "kpi": KPI,
}

In [ ]:
# Salida
import os
os.makedirs(OUT, exist_ok=True)
with open(os.path.join(OUT, "app_data.json"), "w", encoding="utf-8") as f:
    json.dump(A, f, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
print("app_data.json:", round(os.path.getsize(os.path.join(OUT, "app_data.json")) / 1024), "KB")

In [ ]:
# Tabla maestra: cada cifra de APP_DATA (de aquí salen texto, tablas, barras y KPI del HTML)
import json, pandas as pd

def _aplanar(o, ruta=""):
    if isinstance(o, dict):
        for k, v in o.items(): yield from _aplanar(v, f"{ruta}.{k}" if ruta else k)
    elif isinstance(o, list):
        for i, v in enumerate(o): yield from _aplanar(v, f"{ruta}.{i}")
    else:
        yield ruta, o

cifras = pd.DataFrame(list(_aplanar(A)), columns=["ruta", "valor"])
cifras.to_csv(f"{OUT}/cifras_html.csv", index=False)
json.dump(A, open(f"{OUT}/app_data.json", "w"), ensure_ascii=False, indent=1, default=str)
print(len(cifras), "cifras →", f"{OUT}/cifras_html.csv y app_data.json")
cifras.head(40)

In [ ]:
# Verificación contra el HTML publicado (opcional): sube el .html a Colab para correrla
import re, json, os, math
HTML = "dash2.html"
if not os.path.exists(HTML) and IN_COLAB:
    print("Sube", HTML, "para comparar (o ignora esta celda)"); _colab_files.upload()
if os.path.exists(HTML):
    src = open(HTML, encoding="utf-8").read()
    m = re.search(r'<script id="app-data" type="application/json">(.*?)</script>', src, re.S)
    if not m: raise SystemExit("El HTML no incrusta APP_DATA como <script id='app-data'>")
    H = json.loads(m.group(1))
    def _cmp(a, b, ruta, dif):
        if isinstance(a, dict) and isinstance(b, dict):
            for k in a:
                if k in b: _cmp(a[k], b[k], f"{ruta}.{k}" if ruta else k, dif)
        elif isinstance(a, list) and isinstance(b, list):
            for i, (x, y) in enumerate(zip(a, b)): _cmp(x, y, f"{ruta}.{i}", dif)
        elif isinstance(a, (int, float)) and isinstance(b, (int, float)) and not isinstance(a, bool):
            if not math.isclose(a, b, rel_tol=1e-9, abs_tol=1e-9): dif.append((ruta, a, b))
        elif a != b: dif.append((ruta, a, b))
    dif = []; _cmp(A, H, "", dif)
    print("Coinciden todas las cifras comunes" if not dif else f"{len(dif)} diferencias:")
    for d in dif[:50]: print("  ", d)
else:
    print("HTML no encontrado; verificación omitida")